# VeriDoc — Router Fine-tune (DistilBERT, Kaggle/Colab T4)

Fine-tunes a `distilbert-base-uncased` binary classifier to route documents
as `invoice` or `bank_statement` using first-page text extracted by pypdfium2.

**Inputs** (upload to Kaggle dataset before running):
- `router_training_data.csv` — generated by `python training/prepare_router_data.py`

**Outputs** (download after training):
- `router_distilbert/` — HuggingFace model directory (model + tokenizer)
- `router_distilbert.onnx` — ONNX export for CPU inference without torch

**Why DistilBERT over the sklearn TF-IDF baseline?**
- Stronger OOD generalisation on unseen document layouts
- ONNX export means no torch dependency at inference time
- Demonstrates GPU fine-tuning on Kaggle as part of the M3 portfolio story

In [ ]:
!pip install -q transformers datasets scikit-learn optimum[onnxruntime]

In [ ]:
import pandas as pd
from pathlib import Path

df = pd.read_csv('/kaggle/input/veridoc-router/router_training_data.csv')
print(df['label'].value_counts())
print(df.head(2))

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

LABEL2ID = {'bank_statement': 0, 'invoice': 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

df['label_id'] = df['label'].map(LABEL2ID)

train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)
print(f'Train: {len(train_df)}, Val: {len(val_df)}')

train_ds = Dataset.from_pandas(train_df[['text', 'label_id']].rename(columns={'label_id': 'labels'}))
val_ds   = Dataset.from_pandas(val_df[['text', 'label_id']].rename(columns={'label_id': 'labels'}))

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=512, padding='max_length')

train_ds = train_ds.map(tokenize, batched=True, remove_columns=['text'])
val_ds   = val_ds.map(tokenize, batched=True, remove_columns=['text'])
train_ds.set_format('torch')
val_ds.set_format('torch')

In [ ]:
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    accuracy = (preds == labels).mean()
    return {'accuracy': float(accuracy)}

args = TrainingArguments(
    output_dir='router_distilbert',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=10,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='best',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=5,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
# Evaluate on val set
results = trainer.evaluate()
print(results)

# Save model + tokenizer
trainer.save_model('router_distilbert')
tokenizer.save_pretrained('router_distilbert')
print('Saved to router_distilbert/')

In [ ]:
# Export to ONNX (CPU inference — no torch needed downstream)
from optimum.onnxruntime import ORTModelForSequenceClassification

ort_model = ORTModelForSequenceClassification.from_pretrained(
    'router_distilbert', export=True
)
ort_model.save_pretrained('router_distilbert_onnx')
tokenizer.save_pretrained('router_distilbert_onnx')
print('ONNX model saved to router_distilbert_onnx/')

In [ ]:
# Quick sanity check on a sample
from transformers import pipeline as hf_pipeline

pipe = hf_pipeline(
    'text-classification',
    model=ort_model,
    tokenizer=tokenizer,
    device=-1,  # CPU
)

samples = [
    ('Invoice no: 51109301\nSeller: TechVision Distributors', 'invoice'),
    ('Account Statement\nAccount Holders Name TRADE LINKS INDIA', 'bank_statement'),
]
for text, expected in samples:
    pred = pipe(text[:512])[0]
    status = '✓' if pred['label'] == expected else '✗'
    print(f"{status} expected={expected}, got={pred['label']} ({pred['score']:.3f})")

## Results summary

After training, copy the numbers from `trainer.evaluate()` into `eval/REPORT.md` under the `### Router fine-tune lift` section:

| Model | CV Accuracy | Tokens/call | Latency |
|---|---|---|---|
| VLM baseline (LLaMA-4-Scout) | 100.0% | ~2743 | ~2 s |
| TF-IDF + LR (sklearn, local) | 100.0% (5-fold CV) | 0 | <1 ms |
| DistilBERT (this notebook) | ??.?% | 0 | ~50 ms (CPU) |

**Download and place these in `training/artifacts/`:**
- `router_distilbert_onnx/` — ONNX model directory